In [29]:
import numpy as np
import pandas as pd
import random as rd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras.layers import Input, Embedding, Flatten, Dense, Dropout
from tensorflow.keras.models import Model

In [30]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
rd.seed(SEED)

In [31]:
spam = pd.read_csv(r'C:\Users\Jaum\Desktop\CursoIA\7.Processamento de Linguagem Natura\spam.csv')
spam

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [32]:
spam.shape

(5572, 2)

In [33]:
count = spam['Category'].value_counts()
print(count)

Category
ham     4825
spam     747
Name: count, dtype: int64


In [34]:
ham_samples = spam[spam['Category'] == 'ham'].sample(n=747, random_state=42)
spam_samples = spam[spam['Category'] == 'spam']

spam = pd.concat([ham_samples, spam_samples]).sample(frac=1, random_state=42).reset_index(drop=True)

In [35]:
spam.shape

(1494, 2)

In [36]:
labelencoder = LabelEncoder()
y = labelencoder.fit_transform(spam['Category'])

In [37]:
y

array([1, 1, 1, ..., 1, 1, 1])

In [38]:
mensagens = spam['Message'].values
X_train, X_test, y_train, y_test = train_test_split(mensagens, y, test_size=0.3, random_state=42)

In [39]:
token = Tokenizer(num_words=1000)
token.fit_on_texts(X_train)
X_train = token.texts_to_sequences(X_train)
X_test = token.texts_to_sequences(X_test)

In [40]:
print(X_train)

[[32, 3, 15, 28, 157, 415, 15, 857, 1, 218, 25, 74, 1, 102, 15, 596, 61, 39, 696, 697, 11, 308, 143, 108, 37], [341, 170, 36, 19, 416, 279, 20, 417, 171, 144, 25, 117, 1, 512, 10, 280, 513, 342, 28, 63, 103, 280, 698, 858, 281, 418, 78], [117, 699, 4, 71, 1, 68, 2, 118, 202, 7, 309, 9, 31], [2, 61, 29, 859, 219, 220, 79, 700, 29, 128, 31, 64, 72, 310, 12, 597, 219, 220, 282, 80, 65, 158, 159, 13, 145], [8, 16, 3, 514, 515, 75, 9, 598, 13, 221, 73, 22, 8, 109, 48, 75, 189, 111, 419, 75, 516, 19, 51, 239, 6, 18], [2, 24, 262, 283, 32, 7, 28, 284, 23, 42, 72, 6, 222, 468, 860, 420, 263, 861, 12, 37, 145], [112, 517, 41, 240, 39, 223, 57, 5, 224], [69, 50, 24, 190, 1, 73, 8, 203, 99, 138, 38, 2, 16, 55, 3, 225, 47, 85, 6, 23, 204, 124, 36, 191, 226, 37], [183, 227, 131, 50, 30, 74, 701, 264, 702, 11, 16, 703, 518], [343, 519, 599, 184, 1, 32, 17, 9, 38, 104], [90, 344, 285, 12, 380, 345, 600, 344, 862, 380, 345, 11, 468, 344, 863, 864, 863, 100, 346], [9, 3, 865, 41, 17, 172, 51, 381, 1], 

In [41]:
X_train = pad_sequences(X_train, padding='post', maxlen=500)
X_test = pad_sequences(X_test, padding='post', maxlen=500)

In [44]:
input_layer = Input(shape=(500,))
embedding_layer = Embedding(input_dim=(min(1000, len(token.word_index)) + 1), output_dim=50)(input_layer)
flatten_layer = Flatten()(embedding_layer)
dense_layer = Dense(units=10, activation='relu')(flatten_layer)
dropout_layer = Dropout(0.1)(dense_layer)
output_layer = Dense(units=1, activation='sigmoid')(dropout_layer)
modelo = Model(inputs=input_layer, outputs=output_layer)

In [45]:
modelo.compile(loss='mean_squared_error', optimizer='adam', metrics=['accuracy'])
modelo.fit(X_train, y_train, epochs=20, batch_size=10, verbose=True, validation_data=(X_test, y_test))

Epoch 1/20
105/105 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.5270 - loss: 0.2525 - val_accuracy: 0.5033 - val_loss: 0.2441
Epoch 2/20
105/105 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.6099 - loss: 0.2310 - val_accuracy: 0.8886 - val_loss: 0.1737
Epoch 3/20
105/105 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.8684 - loss: 0.1671 - val_accuracy: 0.9220 - val_loss: 0.1456
Epoch 4/20
105/105 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.9114 - loss: 0.1422 - val_accuracy: 0.9332 - val_loss: 0.1302
Epoch 5/20
105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9166 - loss: 0.1282 - val_accuracy: 0.9421 - val_loss: 0.1191
Epoch 6/20
105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9309 - loss: 0.1138 - val_accuracy: 0.9465 - val_loss: 0.1094
Epoch 7/20
105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9396 - loss: 0.1028 - val_accuracy: 0.9510 - val_loss: 0.1013
Epoch 8/20
105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9534 - loss: 0.0916 - val_accu

In [46]:
loss, accuracy = modelo.evaluate(X_test, y_test)
print(f'Loss: {loss}')
print(f'Acurácia: {accuracy}')

15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9665 - loss: 0.0527
Loss: 0.061880819499492645
Acurácia: 0.9510022401809692


In [47]:
nova_previsao = modelo.predict(X_test)
print(nova_previsao[2:5])

15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
[[0.77911973]
 [0.77911973]
 [0.00560704]]


In [48]:
prev = (nova_previsao > 0.5)
print(prev[2:5])

[[ True]
 [ True]
 [False]]


In [50]:
cm = confusion_matrix(y_test, prev)
cm

array([[226,   0],
       [ 22, 201]])